# 🚀 LLM Server via Ollama : Optimisé T4

**Streaming activé · Warmup auto · Changement de modèle facile**

### Avant de démarrer :
1. `Exécution` → `Modifier le type d'exécution` → **GPU T4**
2. Exécuter les cellules dans l'ordre
3. Pour changer de modèle : modifier `MODEL` dans la **Cellule 0** uniquement

In [ ]:
# ════════════════════════════════════════════════════════
# CELLULE 0 — CONFIGURATION GLOBALE
# Modifie uniquement cette cellule pour changer de modèle
# ════════════════════════════════════════════════════════

# ── Catalogue des modèles disponibles ──────────────────
# Décommente le modèle que tu veux utiliser
# Format : nom Ollama, VRAM requise, vitesse relative

MODELS = {
    # ── Ultra-rapides (< 3 GB VRAM) ─────────────────────
    'phi3.5-mini'    : 'phi3.5:mini-instruct-q4_K_M',     # 3.8B  ~2.3 GB
    'gemma2-2b'      : 'gemma2:2b-instruct-q4_K_M',       # 2.6B  ~1.8 GB
    'qwen2.5-3b'     : 'qwen2.5:3b-instruct-q4_K_M',      # 3B    ~2.0 GB

    # ── Équilibrés (3–6 GB VRAM) ────────────────────────
    'mistral-7b'     : 'mistral:7b-instruct-q4_K_M',      # 7B    ~4.1 GB
    'llama3.1-8b'    : 'llama3.1:8b-instruct-q4_K_M',     # 8B    ~4.7 GB
    'qwen2.5-7b'     : 'qwen2.5:7b-instruct-q4_K_M',      # 7B    ~4.1 GB
    'gemma2-9b'      : 'gemma2:9b-instruct-q4_K_M',       # 9B    ~5.4 GB

    # ── Puissants (6–15 GB VRAM T4 max) ────────────────
    'mistral-nemo'   : 'mistral-nemo:12b-instruct-q4_K_M', # 12B  ~7.1 GB
    'llama3.1-14b'   : 'llama3.1:14b-instruct-q4_K_M',    # 14B  ~8.4 GB
    'qwen2.5-14b'    : 'qwen2.5:14b-instruct-q4_K_M',     # 14B  ~8.4 GB
}

# ── CHOIX ACTIF ─────────────────────────────────────────
# Modifie ici : choisir une clé du dict ci-dessus
ACTIVE_MODEL_KEY = 'mistral-7b'

# ── Paramètres de génération ────────────────────────────
MAX_TOKENS   = 512    # Limite de tokens générés (plus petit = modèle plus rapide)
TEMPERATURE  = 0.0    # 0.0 = déterministe, 0.7 = créatif
CTX_SIZE     = 4096   # Taille contexte (plus petit = moins de VRAM)

# ── Résolution ──────────────────────────────────────────
MODEL = MODELS[ACTIVE_MODEL_KEY]

print('━' * 55)
print(f'  Modèle sélectionné : {ACTIVE_MODEL_KEY}')
print(f'  Tag Ollama         : {MODEL}')
print(f'  Max tokens         : {MAX_TOKENS}')
print(f'  Temperature        : {TEMPERATURE}')
print(f'  Contexte           : {CTX_SIZE} tokens')
print('━' * 55)
print()
print('Modèles disponibles :')
for k, v in MODELS.items():
    marker = '  ◀ ACTIF' if k == ACTIVE_MODEL_KEY else ''
    print(f'  {k:<16}  {v}{marker}')

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  Modèle sélectionné : mistral-7b
  Tag Ollama         : mistral:7b-instruct-q4_K_M
  Max tokens         : 512
  Temperature        : 0.0
  Contexte           : 4096 tokens
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Modèles disponibles :
  phi3.5-mini       phi3.5:mini-instruct-q4_K_M
  gemma2-2b         gemma2:2b-instruct-q4_K_M
  qwen2.5-3b        qwen2.5:3b-instruct-q4_K_M
  mistral-7b        mistral:7b-instruct-q4_K_M  ◀ ACTIF
  llama3.1-8b       llama3.1:8b-instruct-q4_K_M
  qwen2.5-7b        qwen2.5:7b-instruct-q4_K_M
  gemma2-9b         gemma2:9b-instruct-q4_K_M
  mistral-nemo      mistral-nemo:12b-instruct-q4_K_M
  llama3.1-14b      llama3.1:14b-instruct-q4_K_M
  qwen2.5-14b       qwen2.5:14b-instruct-q4_K_M


In [ ]:
# ════════════════════════════════════════════════════════
# CELLULE 1 — Vérification GPU
# ════════════════════════════════════════════════════════
import subprocess

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
)

if result.returncode == 0:
    parts = result.stdout.strip().split(',')
    name = parts[0].strip()
    total_mb = int(parts[1].strip().replace(' MiB', ''))
    free_mb  = int(parts[2].strip().replace(' MiB', ''))
    total_gb = total_mb / 1024
    free_gb  = free_mb  / 1024

    print(f'✅ GPU : {name}')
    print(f'   VRAM totale : {total_gb:.1f} GB')
    print(f'   VRAM libre  : {free_gb:.1f} GB')

    # Recommandation selon VRAM disponible
    print()
    if total_gb >= 14:
        print('💡 VRAM suffisante pour les modèles 14B')
    elif total_gb >= 8:
        print('💡 VRAM suffisante pour les modèles 7-12B')
    else:
        print('💡 VRAM limitée — préfère phi3.5-mini ou gemma2-2b pour de la vitesse')
else:
    print('⚠️  Pas de GPU détecté — active le GPU T4 dans les paramètres Colab')
    print('   Exécution → Modifier le type d\'exécution → T4')

✅ GPU : Tesla T4
   VRAM totale : 15.0 GB
   VRAM libre  : 14.6 GB

💡 VRAM suffisante pour les modèles 14B


In [ ]:
# ════════════════════════════════════════════════════════
# CELLULE 2 — Installation Ollama + dépendances (~30s)
# ════════════════════════════════════════════════════════
import subprocess

print('Installation des dépendances...')
subprocess.run(['sudo', 'apt-get', 'install', '-y', '-q', 'zstd'], check=True,
               capture_output=True)
print('  ✅ zstd')

print('Installation Ollama...')
result = subprocess.run(
    'curl -fsSL https://ollama.com/install.sh | sh',
    shell=True, capture_output=True, text=True
)
if result.returncode == 0:
    print('  ✅ Ollama installé')
else:
    print(f'  ❌ Erreur installation : {result.stderr[-300:]}')

# Vérification version
v = subprocess.run(['ollama', '--version'], capture_output=True, text=True)
print(f'  Version : {v.stdout.strip()}')

Installation des dépendances...
  ✅ zstd
Installation Ollama...
  ✅ Ollama installé
  Version : Warning: could not connect to a running Ollama instance


In [ ]:
# ════════════════════════════════════════════════════════
# CELLULE 3 — Démarrage serveur Ollama (optimisé)
# ════════════════════════════════════════════════════════
import subprocess
import os
import time
import requests

def start_ollama():
    # Tue les instances précédentes proprement
    subprocess.run(['pkill', '-9', '-f', 'ollama serve'], capture_output=True)
    time.sleep(2)

    env = os.environ.copy()
    env['OLLAMA_HOST']           = '0.0.0.0:11434'
    env['OLLAMA_KEEP_ALIVE']     = '-1'   # Modèle jamais déchargé de VRAM
    env['OLLAMA_NUM_PARALLEL']   = '1'    # 1 requête à la fois (optimal T4)
    env['OLLAMA_FLASH_ATTENTION'] = '1'   # Flash Attention (gain ~15% vitesse)
    env['OLLAMA_MAX_LOADED_MODELS'] = '1' # 1 seul modèle en VRAM à la fois

    proc = subprocess.Popen(
        ['ollama', 'serve'],
        env=env,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

    print(f'Serveur démarré (PID {proc.pid})')
    print('Attente disponibilité...')

    for i in range(20):
        time.sleep(1)
        try:
            r = requests.get('http://localhost:11434/api/tags', timeout=2)
            if r.status_code == 200:
                print(f'✅ Ollama prêt en {i+1}s')
                return proc
        except:
            pass
    print('⚠️  Timeout — relance cette cellule')
    return None

ollama_proc = start_ollama()

# Vérification port
result = subprocess.run(['ss', '-tlnp'], capture_output=True, text=True)
for line in result.stdout.split('\n'):
    if '11434' in line:
        print(f'Port : {line.strip()}')

Serveur démarré (PID 907)
Attente disponibilité...
✅ Ollama prêt en 1s
Port : LISTEN 0      4096               *:11434            *:*    users:(("ollama",pid=907,fd=3))


In [ ]:
# ════════════════════════════════════════════════════════
# CELLULE 4 — Téléchargement + Warmup du modèle
# ════════════════════════════════════════════════════════
import subprocess
import requests
import time

# ── Vérifie si le modèle est déjà présent ───────────────
r = requests.get('http://localhost:11434/api/tags')
installed = [m['name'] for m in r.json().get('models', [])]

if MODEL in installed:
    print(f'✅ {MODEL} déjà installé — téléchargement ignoré')
else:
    print(f'Téléchargement de {MODEL}...')
    print('(2-6 min selon la connexion Colab)\n')

    process = subprocess.Popen(
        ['ollama', 'pull', MODEL],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )

    last_line = ''
    for line in process.stdout:
        line = line.strip()
        if line and line != last_line:
            # Affiche seulement les lignes de progression utiles
            if any(k in line.lower() for k in ['pulling', 'verif', 'success', 'error', '%']):
                print(f'  {line}')
            last_line = line

    process.wait()

    if process.returncode == 0:
        print(f'\n✅ {MODEL} téléchargé')
    else:
        print(f'\n❌ Erreur (code {process.returncode})')
        raise RuntimeError('Pull échoué — relance cette cellule')

# ── Warmup : charge le modèle en VRAM maintenant ────────
print(f'\nChargement en VRAM (warmup)...')
t0 = time.time()

resp = requests.post('http://localhost:11434/v1/chat/completions', json={
    'model': MODEL,
    'messages': [{'role': 'user', 'content': 'ok'}],
    'max_tokens': 1,
    'temperature': 0.0,
    'options': {'num_ctx': CTX_SIZE}
})

warmup_time = time.time() - t0
if resp.status_code == 200:
    print(f'✅ Modèle chaud en VRAM ({warmup_time:.1f}s) — les appels suivants seront rapides')
else:
    print(f'⚠️  Warmup échoué : {resp.status_code} — {resp.text[:200]}')

# ── VRAM utilisée ────────────────────────────────────────
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=memory.used,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode == 0:
    used, free = result.stdout.strip().split(',')
    used_gb = int(used.strip().replace(' MiB', '')) / 1024
    free_gb = int(free.strip().replace(' MiB', '')) / 1024
    print(f'VRAM : {used_gb:.1f} GB utilisé / {free_gb:.1f} GB libre')

In [ ]:
# ════════════════════════════════════════════════════════
# CELLULE 5 — Test complet avec streaming + benchmark
# ════════════════════════════════════════════════════════
import requests
import json
import time

def chat_stream(messages, system=None, max_tokens=None, temperature=None, verbose=True):
    """Appel streaming avec mesure de performance."""
    msgs = []
    if system:
        msgs.append({'role': 'system', 'content': system})
    msgs.extend(messages)

    payload = {
        'model'      : MODEL,
        'messages'   : msgs,
        'max_tokens' : max_tokens or MAX_TOKENS,
        'temperature': temperature if temperature is not None else TEMPERATURE,
        'stream'     : True,
        'options'    : {'num_ctx': CTX_SIZE},
    }

    t_start = time.time()
    t_first = None
    full_text = ''
    token_count = 0

    resp = requests.post(
        'http://localhost:11434/v1/chat/completions',
        json=payload,
        stream=True
    )

    if verbose:
        print('─' * 50)

    for chunk in resp.iter_lines():
        if not chunk or chunk in (b'data: [DONE]', 'data: [DONE]'):
            continue
        raw = chunk[6:] if chunk[:6] in (b'data: ', 'data: ') else chunk
        try:
            delta = json.loads(raw)['choices'][0]['delta'].get('content', '')
            if delta:
                if t_first is None:
                    t_first = time.time()
                if verbose:
                    print(delta, end='', flush=True)
                full_text += delta
                token_count += 1
        except:
            pass

    t_end = time.time()

    ttft  = (t_first - t_start) if t_first else 0
    total = t_end - t_start
    tps   = token_count / (t_end - t_first) if t_first else 0

    if verbose:
        print(f'\n─' * 25)
        print(f'⏱  TTFT (premier token) : {ttft:.2f}s')
        print(f'⏱  Temps total          : {total:.2f}s')
        print(f'⚡  Débit                : {tps:.1f} tokens/s')
        print(f'📊  Tokens générés       : {token_count}')

    return full_text, {'ttft': ttft, 'total': total, 'tps': tps, 'tokens': token_count}


# ── Test 1 : JSON structuré ──────────────────────────────
print(f'\n🧪 TEST 1 — JSON structuré (modèle : {ACTIVE_MODEL_KEY})')
text1, stats1 = chat_stream(
    messages=[{'role': 'user', 'content': 'Retourne ce JSON exactement: {"intent": "kpi_analysis", "dimensions": ["ca", "marge"], "period": "Q1"}'}],
    system='Tu reponds UNIQUEMENT en JSON valide. Aucun texte autour.',
    max_tokens=100
)

# Valide le JSON
try:
    cleaned = text1.strip().strip('`').replace('json\n', '')
    parsed = json.loads(cleaned)
    print(f'\n✅ JSON valide : {parsed}')
except Exception as e:
    print(f'\n⚠️  JSON non parsable : {e}')


# ── Test 2 : Texte libre ────────────────────────────────
print(f'\n\n🧪 TEST 2 — Génération texte')
text2, stats2 = chat_stream(
    messages=[{'role': 'user', 'content': 'Explique en 3 phrases ce qu\'est un LLM.'}],
    max_tokens=150
)

# ── Résumé comparatif ───────────────────────────────────
print(f'\n\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'  RÉSUMÉ — {ACTIVE_MODEL_KEY} ({MODEL})')
print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
print(f'  Test JSON  — TTFT: {stats1["ttft"]:.2f}s  |  {stats1["tps"]:.1f} tok/s')
print(f'  Test texte — TTFT: {stats2["ttft"]:.2f}s  |  {stats2["tps"]:.1f} tok/s')
avg_tps = (stats1['tps'] + stats2['tps']) / 2
print(f'  Débit moyen : {avg_tps:.1f} tok/s')

if avg_tps > 30:
    print('  🟢 Excellent — fluide en production')
elif avg_tps > 15:
    print('  🟡 Correct — acceptable pour un usage API')
else:
    print('  🔴 Lent — essaie un modèle plus petit (phi3.5-mini)')
print('━' * 55)

In [ ]:
# ════════════════════════════════════════════════════════
# CELLULE 6 — Benchmark multi-modèles (optionnel)
# Compare plusieurs modèles côte à côte
# ════════════════════════════════════════════════════════
import requests
import json
import time
import subprocess

# Modèles à benchmarker (doivent être déjà téléchargés)
# Ajoute/retire des clés selon ce que tu as installé
BENCHMARK_MODELS = [
    'mistral-7b',
    # 'phi3.5-mini',
    # 'llama3.1-8b',
    # 'qwen2.5-7b',
]

BENCH_PROMPT = 'Explique en 2 phrases ce qu\'est le machine learning.'
BENCH_MAX_TOKENS = 100

results = {}

for key in BENCHMARK_MODELS:
    model_tag = MODELS[key]

    # Vérifie que le modèle est installé
    r = requests.get('http://localhost:11434/api/tags')
    installed = [m['name'] for m in r.json().get('models', [])]
    if model_tag not in installed:
        print(f'⏭  {key} non installé — ignoré (lance la cellule 4 avec ce modèle)')
        continue

    print(f'\n🔄 Chargement {key}...')

    # Warmup
    requests.post('http://localhost:11434/v1/chat/completions', json={
        'model': model_tag,
        'messages': [{'role': 'user', 'content': 'ok'}],
        'max_tokens': 1
    })

    # Benchmark
    print(f'⚡ Benchmark {key}...')
    t_start = time.time()
    t_first = None
    token_count = 0
    output = ''

    resp = requests.post(
        'http://localhost:11434/v1/chat/completions',
        json={
            'model': model_tag,
            'messages': [{'role': 'user', 'content': BENCH_PROMPT}],
            'max_tokens': BENCH_MAX_TOKENS,
            'temperature': 0.0,
            'stream': True,
        },
        stream=True
    )

    for chunk in resp.iter_lines():
        if not chunk or b'[DONE]' in chunk:
            continue
        raw = chunk[6:]
        try:
            delta = json.loads(raw)['choices'][0]['delta'].get('content', '')
            if delta:
                if t_first is None:
                    t_first = time.time()
                output += delta
                token_count += 1
        except:
            pass

    t_end = time.time()
    ttft  = (t_first - t_start) if t_first else 0
    tps   = token_count / (t_end - t_first) if t_first else 0

    results[key] = {'ttft': ttft, 'tps': tps, 'tokens': token_count, 'output': output}
    print(f'  ✅ {key}: TTFT={ttft:.2f}s | {tps:.1f} tok/s | {token_count} tokens')

# ── Tableau récapitulatif ────────────────────────────────
if results:
    print(f'\n{"━"*55}')
    print(f'  {"MODÈLE":<18} {"TTFT":>8} {"TOK/S":>8} {"NOTE"}')
    print(f'  {"─"*16} {"─"*8} {"─"*8} {"─"*10}')
    for k, v in sorted(results.items(), key=lambda x: -x[1]['tps']):
        note = '🟢 rapide' if v['tps'] > 30 else ('🟡 correct' if v['tps'] > 15 else '🔴 lent')
        print(f'  {k:<18} {v["ttft"]:>7.2f}s {v["tps"]:>7.1f}  {note}')
    print('━' * 55)

    best = max(results, key=lambda k: results[k]['tps'])
    print(f'\n🏆 Plus rapide : {best} ({results[best]["tps"]:.1f} tok/s)')
    print(f'   → Change ACTIVE_MODEL_KEY = \'{best}\' dans la Cellule 0')


🔄 Chargement mistral-7b...
⚡ Benchmark mistral-7b...
  ✅ mistral-7b: TTFT=0.10s | 37.3 tok/s | 56 tokens

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  MODÈLE                 TTFT    TOK/S NOTE
  ──────────────── ──────── ──────── ──────────
  mistral-7b            0.10s    37.3  🟢 rapide
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

🏆 Plus rapide : mistral-7b (37.3 tok/s)
   → Change ACTIVE_MODEL_KEY = 'mistral-7b' dans la Cellule 0


In [ ]:
# ════════════════════════════════════════════════════════
# CELLULE 7 — Tunnel Cloudflare (accès externe)
# NE PAS FERMER cette cellule tant que le tunnel est actif
# ════════════════════════════════════════════════════════
import subprocess
import time
import re

# Installe cloudflared
print('Installation cloudflared...')
subprocess.run([
    'wget', '-q',
    'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
    '-O', '/usr/local/bin/cloudflared'
], check=True)
subprocess.run(['chmod', '+x', '/usr/local/bin/cloudflared'], check=True)
print('✅ cloudflared installé')

# Lance le tunnel
print('Démarrage du tunnel...')
process = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:11434'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

PUBLIC_URL = None
for line in process.stdout:
    if 'trycloudflare.com' in line:
        match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
        if match:
            PUBLIC_URL = match.group(0)
            break

if PUBLIC_URL:
    print()
    print('═' * 60)
    print('🟢 TUNNEL CLOUDFLARE ACTIF')
    print('═' * 60)
    print(f'URL : {PUBLIC_URL}')
    print()
    print('👉 Variables .env :')
    print(f'   LLM_PROVIDER=local')
    print(f'   LOCAL_LLM_BASE_URL={PUBLIC_URL}')
    print(f'   LOCAL_LLM_MODEL={MODEL}')
    print()
    print('👉 Test curl :')
    print(f'   curl "{PUBLIC_URL}/v1/models"')
    print()
    print('👉 Test streaming curl :')
    print(f'   curl -s "{PUBLIC_URL}/v1/chat/completions" \\')
    print(f'     -H "Content-Type: application/json" \\')
    print(f'     -d \'{{"model":"{MODEL}","messages":[{{"role":"user","content":"Bonjour"}}],"stream":true}}\'')
    print('═' * 60)
    print()
    print('Tunnel actif — ne ferme pas cette cellule.')

    # Keepalive avec statut
    i = 0
    while True:
        time.sleep(60)
        i += 1
        print(f'✅ {time.strftime("%H:%M:%S")} — actif ({i} min) — {PUBLIC_URL}')
else:
    print('❌ URL non trouvée — relance cette cellule')
    process.terminate()

Installation cloudflared...
✅ cloudflared installé
Démarrage du tunnel...

════════════════════════════════════════════════════════════
🟢 TUNNEL CLOUDFLARE ACTIF
════════════════════════════════════════════════════════════
URL : https://healthcare-diary-accessibility-flavor.trycloudflare.com

👉 Variables .env :
   LLM_PROVIDER=local
   LOCAL_LLM_BASE_URL=https://healthcare-diary-accessibility-flavor.trycloudflare.com
   LOCAL_LLM_MODEL=mistral:7b-instruct-q4_K_M

👉 Test curl :
   curl "https://healthcare-diary-accessibility-flavor.trycloudflare.com/v1/models"

👉 Test streaming curl :
   curl -s "https://healthcare-diary-accessibility-flavor.trycloudflare.com/v1/chat/completions" \
     -H "Content-Type: application/json" \
     -d '{"model":"mistral:7b-instruct-q4_K_M","messages":[{"role":"user","content":"Bonjour"}],"stream":true}'
════════════════════════════════════════════════════════════

Tunnel actif — ne ferme pas cette cellule.
✅ 12:17:05 — actif (1 min) — https://healthcare-di